# 🎙️ Meeting Intelligence System — Google Colab Runner

This notebook runs the full **Meeting Intelligence Pipeline** on a free T4 GPU.

**Pipeline Stages:**
1. 🎧 **Whisper** — Speech-to-Text Transcription
2. 👥 **pyannote** — Speaker Diarization
3. 🧠 **DistilBERT + KeyBERT** — Sentiment & Keyword Analysis
4. ✨ **Gemini Pro** — LLM Summarization
5. 🔊 **SpeechT5** — Text-to-Speech Narration

---
### ⚡ Before You Start
1. Go to **Runtime → Change runtime type → T4 GPU** and click Save.
2. You will need your **API Keys** ready (see Step 2 below).
---

## ✅ Step 1: Verify GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

## ✅ Step 2: Set Your API Keys

Enter your keys below. They are stored only in this Colab session and never uploaded anywhere.

In [ ]:
import os
from getpass import getpass

# --- REQUIRED: Google Gemini API Key ---
# Get one for free at: https://aistudio.google.com/
GEMINI_API_KEY = getpass('🔑 Enter your GEMINI_API_KEY: ')

# --- REQUIRED: Hugging Face Token ---
# Get one at: https://huggingface.co/settings/tokens
# Make sure you have accepted terms for:
#   https://huggingface.co/pyannote/speaker-diarization-3.1
#   https://huggingface.co/pyannote/segmentation-3.0
HF_TOKEN = getpass('🔑 Enter your HF_TOKEN: ')

# Set as environment variables
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
os.environ['GEMINI_MODEL']   = 'gemini-1.5-pro'
os.environ['HF_TOKEN']       = HF_TOKEN

print('✅ API keys set successfully!')

## ✅ Step 3: Clone the Repository

In [ ]:
import os

REPO_URL = 'https://github.com/mosomo82/COMP_SCI_5542.git'  # Update if your repo URL changes
PROJECT_DIR = '/content/meeting-intelligence'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print('Repo already cloned. Pulling latest changes...')
    !git -C {PROJECT_DIR} pull

os.chdir(PROJECT_DIR)
print(f'✅ Working directory: {os.getcwd()}')

## ✅ Step 4: Install Dependencies

This installs all required packages. It will take **3–5 minutes** the first time.

In [ ]:
print('Installing system dependencies (ffmpeg)...')
!apt-get install -qq ffmpeg

print('\nInstalling Python packages...')
!pip install -q \
    openai-whisper \
    pyannote.audio>=3.1.0 \
    transformers>=4.37.0 \
    keybert>=0.8.3 \
    sentence-transformers>=2.3.0 \
    google-generativeai>=0.7.2 \
    anthropic>=0.23.0 \
    librosa>=0.10.1 \
    soundfile>=0.12.1 \
    pydub>=0.25.1 \
    python-dotenv>=1.0.0 \
    datasets>=2.16.0

print('\n✅ All packages installed!')

## ✅ Step 5: Upload Your Meeting Audio

Upload any `.mp3`, `.wav`, or `.m4a` file.
Don't have one? We'll download a sample audio clip for you.

In [ ]:
from google.colab import files
import os

USE_SAMPLE = True  # Set to False to upload your own audio file

if USE_SAMPLE:
    # Download a short LibriSpeech sample (clean, multi-speaker)
    !wget -q -O /content/sample_meeting.wav \
        https://www2.cs.uic.edu/~i101/SoundFiles/preamble.wav
    AUDIO_PATH = '/content/sample_meeting.wav'
    print(f'✅ Using sample audio: {AUDIO_PATH}')
else:
    print('Upload your audio file:')
    uploaded = files.upload()
    AUDIO_PATH = '/content/' + list(uploaded.keys())[0]
    print(f'✅ Uploaded: {AUDIO_PATH}')

# Verify the file exists
assert os.path.exists(AUDIO_PATH), f'Audio file not found: {AUDIO_PATH}'
size_mb = os.path.getsize(AUDIO_PATH) / 1024 / 1024
print(f'   File size: {size_mb:.2f} MB')

## ✅ Step 6: Run Stage 1 — Whisper Transcription

Whisper converts your audio to text. Choose `small` for speed, `medium` for accuracy.

In [ ]:
import sys
sys.path.insert(0, '/content/meeting-intelligence')

from src.transcribe import transcribe, save_transcript

WHISPER_MODEL = 'small'  # Options: 'tiny', 'small', 'medium'

print(f'🎧 Transcribing with Whisper ({WHISPER_MODEL})...')
transcription = transcribe(AUDIO_PATH, model_size=WHISPER_MODEL)
save_transcript(transcription)

print('\n' + '='*60)
print('📄 TRANSCRIPT PREVIEW (first 500 chars):')
print('='*60)
print(transcription['text'][:500] + '...')
print(f'\n✅ Total segments: {len(transcription["segments"])}')

## ✅ Step 7: Run Stage 2 — Speaker Diarization

pyannote identifies **who** spoke **when**. This runs on GPU and takes ~10–30 seconds.

> ⚠️ **You must have accepted the terms** for `pyannote/speaker-diarization-3.1` at https://hf.co/pyannote/speaker-diarization-3.1 before running this cell.

In [ ]:
import torch
import whisper
from pyannote.audio import Pipeline
from src.diarize import (
    _align_segments_with_timeline,
    _smooth_short_turn_flips,
    format_diarized_transcript,
    save_diarized,
)

hf_token = os.environ['HF_TOKEN']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Running diarization on: {device}')

print('Loading pyannote pipeline (downloading models on first run)...')
dia_pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    token=hf_token,
)
dia_pipeline.to(device)

# Load audio via Whisper's FFmpeg-based loader (reliable on Colab)
wav_numpy = whisper.load_audio(AUDIO_PATH)
waveform = torch.from_numpy(wav_numpy).unsqueeze(0).to(device)

print('Running diarization...')
diarization = dia_pipeline({'waveform': waveform, 'sample_rate': 16000})

timeline = [
    (turn.start, turn.end, speaker)
    for turn, _, speaker in diarization.itertracks(yield_label=True)
]

diarized = _align_segments_with_timeline(transcription['segments'], timeline)
diarized = _smooth_short_turn_flips(diarized)
save_diarized(diarized)

speakers = set(d['speaker'] for d in diarized)
print(f'\n✅ Detected {len(speakers)} speaker(s): {", ".join(sorted(speakers))}')

diarized_text = format_diarized_transcript(diarized)
print('\n' + '='*60)
print('📄 DIARIZED TRANSCRIPT PREVIEW:')
print('='*60)
print(diarized_text[:800])

## ✅ Step 8: Run Stage 3 — Sentiment & Keyword Analysis

In [ ]:
from src.analyze import analyze_sentiment, extract_keywords, save_analysis

print('🧠 Running sentiment analysis and keyword extraction...')
sentiment = analyze_sentiment(diarized)
keywords  = extract_keywords(transcription['text'])
save_analysis(sentiment, keywords)

print('\n' + '='*60)
print('😊 SENTIMENT RESULTS:')
print('='*60)
for speaker, data in sentiment.items():
    if speaker == 'overall':
        continue
    print(f'  {speaker}: {data["label"]} ({data["score"]:.0%})')
overall = sentiment.get('overall', {})
print(f'  Overall: {overall.get("label", "N/A")} ({overall.get("score", 0):.0%})')

print('\n🔑 TOP KEYWORDS:')
print('  ' + ', '.join(keywords))
print('\n✅ Analysis complete!')

## ✅ Step 9: Run Stage 4 — Gemini LLM Summarization

Choose between `baseline` and `improved` prompts for your evaluation comparison.

In [ ]:
from src.summarize import summarize, format_summary_for_speech, save_summary

PROMPT_VARIANT = 'improved'  # Options: 'baseline', 'improved'

print(f'✨ Generating summary with Gemini (prompt: {PROMPT_VARIANT})...')
summary = summarize(
    diarized_transcript=diarized_text,
    keywords=keywords,
    sentiment=sentiment,
    prompt_variant=PROMPT_VARIANT,
)
save_summary(summary)
summary_text = format_summary_for_speech(summary)

print('\n' + '='*60)
print('📋 STRUCTURED SUMMARY:')
print('='*60)
if summary.get('executive_summary'):
    print(f'\n📌 Executive Summary:\n{summary["executive_summary"]}')
if summary.get('key_decisions'):
    print('\n🎯 Key Decisions:')
    for d in summary['key_decisions']:
        print(f'  • {d}')
if summary.get('action_items'):
    print('\n✅ Action Items:')
    for item in summary['action_items']:
        deadline = f" (due: {item['deadline']})" if item.get('deadline') else ''
        print(f'  • {item.get("owner", "?")}: {item.get("task", "")}{deadline}')
print('\n✅ Summary complete!')

## ✅ Step 10: Run Stage 5 — SpeechT5 Text-to-Speech (Optional)

Generates a voiced audio narration of the summary. Skip this if you only care about text results.

In [ ]:
from IPython.display import Audio, display
from src.speak import synthesize_speech

GENERATE_AUDIO = True  # Set to False to skip this stage

if GENERATE_AUDIO:
    print('🔊 Synthesizing voiced summary with SpeechT5...')
    audio_out_path = synthesize_speech(summary_text)
    print(f'✅ Audio saved to: {audio_out_path}')
    print('\n🎧 Playing voiced summary:')
    display(Audio(audio_out_path, autoplay=False))
else:
    audio_out_path = None
    print('⏭️  TTS skipped.')

## ✅ Step 11: Run the FULL Pipeline at Once

Once all individual stages are confirmed working, use this cell to run everything in one shot.

In [ ]:
import os, sys
sys.path.insert(0, '/content/meeting-intelligence')
os.chdir('/content/meeting-intelligence')

# Make sure env vars are set (re-run Step 2 if needed)
from src.pipeline import run_pipeline

result = run_pipeline(
    audio_path     = AUDIO_PATH,
    whisper_model  = 'small',       # 'tiny' | 'small' | 'medium'
    prompt_variant = 'improved',    # 'baseline' | 'improved'
    generate_audio = True,
)

print('\n' + '='*60)
print('🏁 PIPELINE COMPLETE — LATENCY BREAKDOWN:')
print('='*60)
for stage, secs in result['stage_times'].items():
    print(f'  {stage:<20} {secs}s')
print(f'  {"TOTAL":<20} {sum(result["stage_times"].values())}s')

# Play voiced summary
if result.get('audio_path'):
    from IPython.display import Audio, display
    display(Audio(result['audio_path'], autoplay=False))

## ✅ Step 12: Download All Outputs

Downloads everything from the `outputs/` folder to your local machine.

In [ ]:
import shutil
from google.colab import files
import os

OUTPUT_DIR = '/content/meeting-intelligence/outputs'
ZIPFILE    = '/content/meeting_outputs.zip'

shutil.make_archive('/content/meeting_outputs', 'zip', OUTPUT_DIR)
print(f'📦 Zipped outputs: {ZIPFILE}')

files.download(ZIPFILE)
print('✅ Download started!')

print('\n📂 Files in outputs/:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}  ({size/1024:.1f} KB)')

## 🎓 Step 13: Evaluation — Baseline vs Improved (For Grading)

Runs the **same audio** through both prompt variants and compares the output quality.

In [ ]:
import time
from src.summarize import summarize

print('Running BASELINE prompt...')
t0 = time.time()
summary_baseline = summarize(
    diarized_transcript=diarized_text,
    keywords=keywords,
    sentiment=sentiment,
    prompt_variant='baseline',
)
baseline_time = round(time.time() - t0, 2)

print('Running IMPROVED prompt...')
t0 = time.time()
summary_improved = summarize(
    diarized_transcript=diarized_text,
    keywords=keywords,
    sentiment=sentiment,
    prompt_variant='improved',
)
improved_time = round(time.time() - t0, 2)

print('\n' + '='*60)
print('📊 COMPARISON RESULTS')
print('='*60)

print('\n🔵 BASELINE SUMMARY:')
print(summary_baseline.get('executive_summary', 'N/A'))
print(f'  Action items: {len(summary_baseline.get("action_items", []))}')
print(f'  Latency: {baseline_time}s')

print('\n🟢 IMPROVED SUMMARY:')
print(summary_improved.get('executive_summary', 'N/A'))
print(f'  Action items: {len(summary_improved.get("action_items", []))}')
print(f'  Latency: {improved_time}s')

print('\n✅ Use this comparison in your evaluation notebook!')